In [ ]:
# ============================================================================
# CÀI ĐẶT + TẢI MODEL ĐÃ TRAIN SẴN (notebook này KHÔNG train lại, chỉ để nhận diện video)
# ============================================================================
!pip install -q ultralytics

import os
import numpy as np
from ultralytics import YOLO
from tensorflow.keras.models import load_model

YOLO_WEIGHTS = 'runs/detect/train/weights/best.pt'   # ⭐ sửa nếu bạn lưu weight ở chỗ khác (vd Kaggle Model/Dataset đính kèm)
CNN_WEIGHTS = '/kaggle/working/char_cnn.h5'           # ⭐ sửa nếu bạn lưu weight ở chỗ khác

if not os.path.exists(YOLO_WEIGHTS):
    raise FileNotFoundError(
        f"Không tìm thấy weight YOLO: {YOLO_WEIGHTS}\n"
        f"-> Notebook này không train lại, cần Add Input dataset/model đã chứa best.pt, "
        f"hoặc sửa lại YOLO_WEIGHTS cho đúng đường dẫn."
    )
if not os.path.exists(CNN_WEIGHTS):
    raise FileNotFoundError(
        f"Không tìm thấy weight CNN: {CNN_WEIGHTS}\n"
        f"-> Notebook này không train lại, cần Add Input dataset/model đã chứa char_cnn.h5, "
        f"hoặc sửa lại CNN_WEIGHTS cho đúng đường dẫn."
    )

detector = YOLO(YOLO_WEIGHTS)
cnn = load_model(CNN_WEIGHTS)

labels = sorted(['0','1','2','3','4','5','6','7','8','9',
                 'A','B','C','D','E','F','G','H','K','L','M','N',
                 'P','R','S','T','U','V','X','Y','Z','Noise'])
IDX_TO_CHAR = {i: c for i, c in enumerate(labels)}

print("✅ Đã tải xong detector (YOLO) và model đọc ký tự (CNN)")

In [ ]:
# ============================================================================
# CÁC HÀM XỬ LÝ: tách hàng, tách ký tự, đọc ký tự, sửa nhầm số/chữ theo ngữ pháp biển
# (gộp lại từ các bản fix trước, dùng chung cho cả ảnh lẫn video)
# ============================================================================
import cv2
import numpy as np

def deskew_plate(plate_bgr):
    """⭐ Làm thẳng biển bị nghiêng/xoay trước khi tách ký tự.
    Biển chụp xéo (~10-20°) là nguyên nhân chính khiến segment hoàn toàn
    thất bại (trả về "?")."""
    gray = cv2.cvtColor(plate_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    thr = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    coords = cv2.findNonZero(thr)
    if coords is None:
        return plate_bgr
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = 90 + angle
    # Chỉ sửa khi lệch vừa phải (tránh xoay sai khi ảnh đã thẳng hoặc quá nhiễu)
    if abs(angle) < 1 or abs(angle) > 20:
        return plate_bgr
    h, w = plate_bgr.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(plate_bgr, M, (w, h), flags=cv2.INTER_CUBIC,
                           borderMode=cv2.BORDER_REPLICATE)


def find_row_split(thr):
    """⭐ Tìm hàng ít mực nhất ở giữa ảnh để tách 2 hàng,
    thay vì tỷ lệ cố định 0.55/0.45 (cắt xuyên ký tự khi tỷ lệ 2 hàng
    khác chuẩn, làm mất ký tự đầu/cuối)."""
    h = thr.shape[0]
    lo, hi = int(h * 0.3), int(h * 0.7)
    if hi <= lo:
        return h // 2
    row_ink = thr[lo:hi, :].sum(axis=1)
    return lo + int(np.argmin(row_ink))


def _iou(b1, b2):
    x1, y1, w1, h1 = b1
    x2, y2, w2, h2 = b2
    xa, ya = max(x1, x2), max(y1, y2)
    xb, yb = min(x1 + w1, x2 + w2), min(y1 + h1, y2 + h2)
    inter = max(0, xb - xa) * max(0, yb - ya)
    union = w1 * h1 + w2 * h2 - inter
    return inter / union if union > 0 else 0


MIN_CHAR_CONF = 0.55  # ⭐ bỏ ký tự đoán độ tin cậy thấp thay vì nhận bừa

# ⭐ bảng nhầm lẫn 2 chiều số<->chữ có hình dạng giống nhau (CNN hay lẫn)
DIGIT_LETTER_CONFUSION = {
    '0': 'D', 'D': '0',
    '2': 'Z', 'Z': '2',
    '5': 'S', 'S': '5',
    '6': 'G', 'G': '6',
    '8': 'B', 'B': '8',
}

def fix_plate_grammar(s):
    """⭐ Biển ô tô 1 hàng VN chuẩn luôn có dạng SỐ-SỐ-CHỮ-SỐ-SỐ-SỐ-SỐ-SỐ (8 ký tự).
    Dùng đúng vị trí ký tự để sửa các cặp số/chữ hay bị CNN đọc nhầm hình dạng
    (2/Z, 8/B, 0/D, 5/S, 6/G) mà không cần train lại model."""
    if len(s) != 8:
        return s
    out = list(s)
    for i, ch in enumerate(out):
        expect_digit = (i != 2)
        swap = DIGIT_LETTER_CONFUSION.get(ch)
        if swap is None:
            continue
        if expect_digit and not ch.isdigit() and swap.isdigit():
            out[i] = swap
        elif not expect_digit and ch.isdigit() and not swap.isdigit():
            out[i] = swap
    return ''.join(out)


def _char_boxes_from_thr(thr, H, W):
    cnts, _ = cv2.findContours(thr, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in cnts:
        x,y,w,h = cv2.boundingRect(c)
        hr = h/float(H)
        arr = (w*h)/float(H*W)
        if not (0.35<hr<0.95 and arr>0.008):
            continue
        ar = w/float(h) if h>0 else 0
        if ar >= 2.4:
            continue

        split = None
        if ar > 0.75:
            # ⭐ luôn kiểm tra khe hở thật trước khi coi là 1 ký tự rộng hay 2 ký tự dính
            #    (chỉ xét khi ar>=1.1 thì cặp số hẹp dính sát như "29" sẽ bị bỏ sót)
            sub = thr[y:y+h, x:x+w]
            col_ink = sub.sum(axis=0)
            lo_c, hi_c = int(w*0.25), int(w*0.75)
            if hi_c > lo_c:
                seg = col_ink[lo_c:hi_c]
                valley = lo_c + int(np.argmin(seg))
                typical = np.median(col_ink[col_ink > 0]) if np.any(col_ink > 0) else 0
                if typical > 0 and col_ink[valley] < 0.25 * typical:
                    left_w, right_w = valley, w - valley
                    left_ar = left_w/float(h)
                    right_ar = right_w/float(h)
                    if 0.08<left_ar<1.1 and 0.08<right_ar<1.1 and left_w > 2 and right_w > 2:
                        split = ((x, y, left_w, h), (x+valley, y, right_w, h))

        if split:
            boxes.append(split[0])
            boxes.append(split[1])
        elif 0.08<ar<1.1:               # ⭐ hạ ngưỡng dưới 0.15->0.08, số "1" mảnh không còn bị loại
            boxes.append((x,y,w,h))
        # else: ar>=1.1 nhưng không có khe hở rõ -> bỏ qua
    boxes = sorted(boxes, key=lambda b: b[2]*b[3], reverse=True)
    kept=[]
    for cand in boxes:
        x,y,w,h=cand; cx,cy=x+w/2,y+h/2
        overlap = False
        for kb in kept:
            kx,ky,kw,kh = kb
            if (kx<=cx<=kx+kw and ky<=cy<=ky+kh) or _iou(cand, kb) > 0.3:   # ⭐ thêm kiểm tra IoU
                overlap = True
                break
        if not overlap:
            kept.append(cand)
    return sorted(kept, key=lambda b: b[0])


def _binarize_otsu(gray):
    return cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]


def _binarize_adaptive(gray):
    """⭐ phương án dự phòng khi Otsu thất bại (biển loá sáng/sáng không đều)."""
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    eq = clahe.apply(gray)
    return cv2.adaptiveThreshold(eq, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY_INV, 25, 8)


def extract_chars(plate_bgr):
    plate_bgr = deskew_plate(plate_bgr)          # ⭐ làm thẳng biển trước
    h0 = plate_bgr.shape[0]
    if h0 < 120:
        s = 120 / h0
        plate_bgr = cv2.resize(plate_bgr, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(plate_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3,3), 0)
    H, W = gray.shape

    # ⭐ xóa viền/khung biển để tránh lẫn ốc vít, mép kim loại thành "ký tự" giả
    b = max(2, int(0.015 * min(H, W)))

    kept, thr = [], None
    for binarize in (_binarize_otsu, _binarize_adaptive):  # ⭐ Otsu thất bại (0 ký tự) -> thử adaptive+CLAHE
        cand_thr = binarize(gray)
        cand_thr[:b, :] = 0; cand_thr[-b:, :] = 0; cand_thr[:, :b] = 0; cand_thr[:, -b:] = 0
        cand_kept = _char_boxes_from_thr(cand_thr, H, W)
        if cand_kept:
            kept, thr = cand_kept, cand_thr
            break

    if not kept:
        return []

    chars=[]
    for (x,y,w,h) in kept:
        crop = thr[y:y+h, x:x+w]
        size=max(w,h)
        sq=np.zeros((size,size),dtype='uint8')
        sq[(size-h)//2:(size-h)//2+h,(size-w)//2:(size-w)//2+w]=crop
        chars.append(cv2.resize(sq,(28,28)))
    return chars

def predict_str(imgs):
    if not imgs: return ""
    batch=np.array(imgs).reshape(-1,28,28,1).astype('float32')/255.0
    preds=cnn.predict(batch, verbose=0)
    s=""
    for p in preds:
        ch=IDX_TO_CHAR[int(np.argmax(p))]
        conf = float(np.max(p))
        if ch!='Noise' and conf >= MIN_CHAR_CONF: s+=ch   # ⭐
    return s

def read_plate(plate_bgr, ptype):
    if ptype==1:  # 2 hàng
        gray = cv2.cvtColor(plate_bgr, cv2.COLOR_BGR2GRAY)
        thr_full = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        split = find_row_split(thr_full)                 # ⭐ tách theo khe hở thực tế
        margin = max(2, int(0.03 * plate_bgr.shape[0]))
        return predict_str(extract_chars(plate_bgr[:split + margin,:])) + \
               predict_str(extract_chars(plate_bgr[max(0, split - margin):,:]))
    return fix_plate_grammar(predict_str(extract_chars(plate_bgr)))   # ⭐ áp luật vị trí số/chữ cho biển 1 hàng

print("✅ Đã định nghĩa xong các hàm xử lý")

In [ ]:
# ============================================================================
# NHẬN DIỆN BIỂN SỐ TỪ VIDEO
# ============================================================================
import os, cv2

VIDEO_PATH = '/kaggle/input/datasets/nahidwin2k11/nahnah/video.mp4'   # ⭐ sửa lại đúng đường dẫn video của bạn
OUTPUT_VIDEO = '/kaggle/working/result_video.mp4'
DETECT_EVERY_N_FRAMES = 5     # ⭐ tăng số này nếu video chạy chậm (không cần detect lại mỗi khung hình)
CONF_THRESHOLD = 0.5
DETECT_IMGSZ = 1280            # ⭐ tăng so với mặc định 640 để bắt biển nhỏ/ở xa tốt hơn (đổi lại chậm hơn)

# ⭐ Che các vùng cố định KHÔNG phải biển số (vd dòng ngày giờ camera đóng dấu) để
#    tránh detect nhầm thành biển số. Mỗi vùng là (x1, y1, x2, y2) theo pixel của
#    khung hình gốc. Để trống [] nếu video không có watermark/timestamp cố định.
MASK_REGIONS = [
    # (0, 0, 400, 40),   # ví dụ: che góc trên-trái nếu camera đóng dấu ngày giờ ở đó
]

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(
        f"Không tìm thấy video: {VIDEO_PATH}\n"
        f"-> Kiểm tra: (1) dataset chứa video đã được Add Input chưa, "
        f"(2) tên file/đường dẫn có đúng chính tả không."
    )

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Không mở được video (có thể sai định dạng/codec): {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"🎬 Video: {frame_w}x{frame_h} @ {fps:.1f}fps, {total_frames} khung hình")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (frame_w, frame_h))

# ⭐ Giữ lại nhãn của lần detect gần nhất để hiển thị ở các khung hình bị bỏ qua
#    (tránh khung/chữ nhấp nháy biến mất liên tục khi không detect lại mỗi frame)
last_boxes = []   # [(x1, y1, x2, y2, label), ...]
MISS_TOLERANCE = 3   # ⭐ cho phép trượt tối đa 3 lần detect liên tiếp mà vẫn giữ khung cũ,
                      #    thay vì xoá khung ngay lần đầu không thấy (gây chớp tắt liên tục
                      #    khi model chỉ bắt trúng biển may rủi giữa các khung hình)
miss_streak = 0

frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break

    if frame_idx % DETECT_EVERY_N_FRAMES == 0:
        detect_frame = frame
        if MASK_REGIONS:                       # ⭐ che vùng watermark trước khi detect
            detect_frame = frame.copy()
            for (mx1, my1, mx2, my2) in MASK_REGIONS:
                detect_frame[my1:my2, mx1:mx2] = 0

        results = detector.predict(detect_frame, conf=CONF_THRESHOLD, iou=0.45,
                                    imgsz=DETECT_IMGSZ, agnostic_nms=True, verbose=False)[0]  # ⭐ imgsz lớn hơn
        new_boxes = []
        if results.boxes is not None:
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls_id = int(box.cls[0])
                crop = frame[y1:y2, x1:x2]
                plate_str = read_plate(crop, cls_id)
                new_boxes.append((x1, y1, x2, y2, plate_str if plate_str else "?"))

        if new_boxes:
            last_boxes = new_boxes
            miss_streak = 0
        else:
            miss_streak += 1
            if miss_streak > MISS_TOLERANCE:   # ⭐ trượt quá lâu mới thật sự xoá khung
                last_boxes = []

    for (x1, y1, x2, y2, label) in last_boxes:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 2)
        cv2.rectangle(frame, (x1, max(0, y1 - th - 12)), (x1 + tw + 10, y1), (0, 255, 0), -1)
        cv2.putText(frame, label, (x1 + 5, max(15, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2)

    writer.write(frame)
    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"   ...đã xử lý {frame_idx}/{total_frames} khung hình")

cap.release()
writer.release()
print(f"\n✅ Xong! Video kết quả lưu tại: {OUTPUT_VIDEO}")

# Xem trước ngay trong notebook
from IPython.display import Video
Video(OUTPUT_VIDEO, embed=True, width=640)